# FoodLens Expanded Taxonomy V1 Baseline

This notebook trains the first classifier that can predict more than the original 101 Food-101 classes. It uses the conservative expanded taxonomy from `expanded_taxonomy_v1.json`: Food-101 plus audited public-dataset labels after spelling cleanup and obvious Food-101 mappings.

- Start from the A3b ConvNeXt-Tiny checkpoint.
- Replace the 101-class head with an expanded taxonomy head.
- Freeze the backbone for this first baseline so the run tests taxonomy viability before full fine-tuning.
- Keep Food-101 evaluation visible as a regression check.
- Treat this as an expansion baseline, not a product champion.

## 1. Experiment Contract

- **Run:** `E1` expanded taxonomy v1 baseline.
- **Target classes:** Food-101 plus audited new public-dataset classes.
- **Starting point:** A3b ConvNeXt-Tiny checkpoint when available.
- **Training scope:** classifier head only for the first baseline.
- **Datasets:** Food-101, Food Image Classification Dataset, and Foodies.AI Challenge.
- **Outputs:** taxonomy, manifests, training history, validation/test metrics, calibration summary, class reports, source-dataset metrics, and best checkpoint.

The purpose is to answer whether the expanded taxonomy is trainable and coherent before moving to FoodX-251 or full-backbone fine-tuning.

## 2. Imports And Runtime Setup

- Keep the PyTorch bootstrap before importing `torch` to handle Kaggle P100 assignments.
- Read the taxonomy JSON from the Kaggle package or local repo path.
- Use the same image normalization as the A3b ConvNeXt model.

In [ ]:
import json
import os
import random
import re
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


def ensure_cuda_compatible_torch() -> None:
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        return

    compute_capability = result.stdout.strip().splitlines()[0] if result.stdout.strip() else ""
    if compute_capability != "6.0" or os.getenv("FOODLENS_TORCH_BOOTSTRAPPED") == "1":
        return

    print("Detected CUDA compute capability 6.0. Installing compatible torch build.")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--upgrade",
            "torch==2.5.1",
            "torchvision==0.20.1",
            "--index-url",
            "https://download.pytorch.org/whl/cu121",
        ]
    )
    os.environ["FOODLENS_TORCH_BOOTSTRAPPED"] = "1"


ensure_cuda_compatible_torch()

import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}")

## 3. Configuration

- `expanded_taxonomy_v1.json` controls label cleanup and excluded raw labels.
- The first run caps per-class samples to reduce runtime and avoid over-weighting large public classes.
- A3b remains the Food-101 accuracy reference for regression checks.

In [ ]:
@dataclass(frozen=True)
class CFG:
    RUN_ID: str = "e1_expanded_taxonomy_v1_head_224"
    SEED: int = 42
    IMAGE_SIZE: tuple[int, int] = (224, 224)
    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 2
    MAX_EPOCHS: int = 5
    PATIENCE: int = 2
    HEAD_LR: float = 5e-4
    WEIGHT_DECAY: float = 1e-2
    LABEL_SMOOTHING: float = 0.05
    TOP_K: int = 5
    ECE_BINS: int = 15
    MAX_IMAGES_PER_CLASS: int = 1000
    RESULTS_ROOT: Path = Path("/kaggle/working/results/expanded_taxonomy")
    FOOD101_TEST_TOP_1: float = 83.90098810195923
    FOOD101_TEST_TOP_5: float = 95.7821786403656
    FOOD101_ECE: float = 0.055595677345991135

RESULTS_DIR = CFG.RESULTS_ROOT / CFG.RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


EMBEDDED_TAXONOMY_CFG = {'name': 'foodlens_expanded_taxonomy_v1', 'description': 'Conservative expanded taxonomy built from Food-101 plus audited public Kaggle food datasets. FoodX-251 remains preferred for the long-term expansion, but it requires Kaggle competition access first.', 'baseline': {'product_champion': 'ResNet50 FT-V2', 'accuracy_leader': 'A3b ConvNeXt-Tiny continued fine-tune', 'food101_test_top_1': 83.90098810195923, 'food101_test_top_5': 95.7821786403656, 'food101_ece_calibrated': 0.055595677345991135}, 'min_images_for_new_class': 100, 'raw_label_to_canonical': {'burger': 'hamburger', 'chapathi': 'chapati', 'chole_bature': 'chole_bhature', 'donut': 'donuts', 'fries': 'french_fries', 'pakode': 'pakoda', 'taco': 'tacos'}, 'exclude_raw_labels': {'aloo_matar': 'Only 97 images in the audited public dataset; hold until more examples are available.'}, 'candidate_new_classes': ['baked_potato', 'besan_cheela', 'biryani', 'butter_naan', 'chai', 'chapati', 'chole_bhature', 'crispy_chicken', 'dahl', 'dal_makhani', 'dhokla', 'dosa', 'gulab_jamun', 'idli', 'jalebi', 'kaathi_rolls', 'kadai_paneer', 'kulfi', 'masala_dosa', 'momos', 'naan', 'paani_puri', 'pakoda', 'pav_bhaji', 'poha', 'rolls', 'sandwich', 'taquito', 'vada_pav'], 'known_access_blockers': {'ifood-2019-fgvc6': 'FoodX-251 is visible but not downloadable until the Kaggle account enters the community competition.'}}


def load_taxonomy_config() -> dict:
    candidates = [
        Path("expanded_taxonomy_v1.json"),
        Path("configs/expanded_taxonomy_v1.json"),
        Path("../configs/expanded_taxonomy_v1.json"),
        Path("/kaggle/working/expanded_taxonomy_v1.json"),
    ]
    for candidate in candidates:
        if candidate.exists():
            print(f"Loaded taxonomy config: {candidate}")
            return json.loads(candidate.read_text())
    print("Using embedded expanded taxonomy config.")
    return EMBEDDED_TAXONOMY_CFG

TAXONOMY_CFG = load_taxonomy_config()
RAW_TO_CANONICAL = TAXONOMY_CFG["raw_label_to_canonical"]
EXCLUDED_RAW_LABELS = set(TAXONOMY_CFG["exclude_raw_labels"].keys())
CANDIDATE_NEW_CLASSES = set(TAXONOMY_CFG["candidate_new_classes"])
(RESULTS_DIR / "expanded_taxonomy_v1.json").write_text(json.dumps(TAXONOMY_CFG, indent=2))

## 4. Dataset Roots And Label Mapping

- Resolve both Kaggle mount patterns: `/kaggle/input/datasets/...` and `/kaggle/input/<slug>`.
- Keep Food-101 labels as benchmark labels.
- Map only explicit spelling/singular/plural fixes from the taxonomy config.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

@dataclass(frozen=True)
class DatasetSpec:
    name: str
    root_candidates: tuple[Path, ...]
    class_container_candidates: tuple[str, ...]

DATASETS = [
    DatasetSpec(
        name="food101",
        root_candidates=(Path("/kaggle/input/datasets/kmader/food41"), Path("/kaggle/input/food41")),
        class_container_candidates=("images", ""),
    ),
    DatasetSpec(
        name="food_image_classification",
        root_candidates=(
            Path("/kaggle/input/datasets/harishkumardatalab/food-image-classification-dataset"),
            Path("/kaggle/input/food-image-classification-dataset"),
        ),
        class_container_candidates=("Food Classification dataset", ""),
    ),
    DatasetSpec(
        name="foodies_ai_challenge",
        root_candidates=(
            Path("/kaggle/input/datasets/jvageesh11/foodies-ai-food-image-classification-challenge"),
            Path("/kaggle/input/foodies-ai-food-image-classification-challenge"),
        ),
        class_container_candidates=("Foodies_Challenge", ""),
    ),
]


def normalize_label(label: str) -> str:
    label = label.strip().lower()
    label = re.sub(r"[()\[\]{}]", " ", label)
    label = re.sub(r"[^a-z0-9]+", "_", label)
    return re.sub(r"_+", "_", label).strip("_")


def canonical_label(raw_label: str) -> Optional[str]:
    normalized = normalize_label(raw_label)
    if normalized in EXCLUDED_RAW_LABELS:
        return None
    return RAW_TO_CANONICAL.get(normalized, normalized)


def resolve_class_container(spec: DatasetSpec) -> Optional[Path]:
    for root in spec.root_candidates:
        if not root.exists():
            continue
        for relative in spec.class_container_candidates:
            container = root / relative if relative else root
            if container.exists() and any(path.is_dir() for path in container.iterdir()):
                return container
    return None

for spec in DATASETS:
    print(spec.name, resolve_class_container(spec))

## 5. Expanded Manifest

- Build one manifest with canonical labels.
- Cap each class to a fixed maximum for a faster first baseline.
- Export raw and canonical label counts so taxonomy decisions remain auditable.

In [ ]:
def build_manifest_for_dataset(spec: DatasetSpec) -> pd.DataFrame:
    container = resolve_class_container(spec)
    if container is None:
        raise FileNotFoundError(f"Could not resolve class container for {spec.name}")

    records = []
    for class_dir in sorted(path for path in container.iterdir() if path.is_dir()):
        raw_label = class_dir.name
        label = canonical_label(raw_label)
        if label is None:
            continue
        image_paths = sorted(
            path for path in class_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        if len(image_paths) > CFG.MAX_IMAGES_PER_CLASS:
            stable_offset = sum(ord(ch) for ch in f"{spec.name}:{label}")
            rng = random.Random(CFG.SEED + stable_offset)
            image_paths = sorted(rng.sample(image_paths, CFG.MAX_IMAGES_PER_CLASS))
        for image_path in image_paths:
            records.append(
                {
                    "path": str(image_path),
                    "dataset": spec.name,
                    "raw_label": raw_label,
                    "label": label,
                }
            )
    return pd.DataFrame.from_records(records)

manifest_df = pd.concat([build_manifest_for_dataset(spec) for spec in DATASETS], ignore_index=True)
class_counts_df = (
    manifest_df.groupby(["label"])
    .agg(image_count=("path", "count"), source_datasets=("dataset", lambda values: sorted(set(values))))
    .reset_index()
    .sort_values("label")
)
class_names = sorted(class_counts_df["label"].tolist())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

manifest_df.to_csv(RESULTS_DIR / "expanded_manifest.csv", index=False)
class_counts_df.to_csv(RESULTS_DIR / "expanded_class_counts.csv", index=False)
(RESULTS_DIR / "expanded_class_names.json").write_text(json.dumps(class_names, indent=2))

print(f"Images: {len(manifest_df):,}")
print(f"Classes: {len(class_names):,}")
print(f"New classes beyond Food-101 config: {len(CANDIDATE_NEW_CLASSES):,}")
display(class_counts_df.head(20))
display(class_counts_df.tail(20))

## 6. Stratified Split And Loaders

- Use a stratified split across the expanded label set.
- Use weighted sampling in training so smaller new classes are not overwhelmed by Food-101 classes.
- Keep validation/test deterministic.

In [ ]:
train_df, temp_df = train_test_split(
    manifest_df,
    test_size=0.2,
    random_state=CFG.SEED,
    stratify=manifest_df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=CFG.SEED,
    stratify=temp_df["label"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(RESULTS_DIR / "train_manifest.csv", index=False)
val_df.to_csv(RESULTS_DIR / "val_manifest.csv", index=False)
test_df.to_csv(RESULTS_DIR / "test_manifest.csv", index=False)

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]
TRAIN_TRANSFORMS = transforms.Compose([
    transforms.RandomResizedCrop(CFG.IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])
EVAL_TRANSFORMS = transforms.Compose([
    transforms.Resize(CFG.IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

class FoodDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform: transforms.Compose) -> None:
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        row = self.df.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        return self.transform(image), class_to_idx[row["label"]]

train_label_counts = train_df["label"].value_counts().to_dict()
train_weights = train_df["label"].map(lambda label: 1.0 / train_label_counts[label]).to_numpy()
sampler = WeightedRandomSampler(weights=train_weights, num_samples=len(train_weights), replacement=True)

train_loader = DataLoader(
    FoodDataset(train_df, TRAIN_TRANSFORMS),
    batch_size=CFG.BATCH_SIZE,
    sampler=sampler,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
val_loader = DataLoader(
    FoodDataset(val_df, EVAL_TRANSFORMS),
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    FoodDataset(test_df, EVAL_TRANSFORMS),
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=device.type == "cuda",
)

print(f"Train/val/test: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")

## 7. Model Initialization

- Build the same ConvNeXt-Tiny head structure as A3b, but with the expanded class count.
- Load all compatible A3b weights and skip the final 101-class classifier layer.
- Freeze the backbone and train the classifier for this first baseline.

In [ ]:
def make_classifier_head(in_features: int, num_classes: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, num_classes),
    )


def build_convnext_tiny(num_classes: int) -> nn.Module:
    model = models.convnext_tiny(weights=None)
    in_features = model.classifier[2].in_features
    model.classifier[2] = make_classifier_head(in_features, num_classes)
    return model


def resolve_a3b_checkpoint() -> Optional[Path]:
    matches = sorted(Path("/kaggle/input").rglob("convnext_tiny_continued_best.pth"))
    return matches[0] if matches else None


def load_compatible_weights(model: nn.Module, checkpoint_path: Path) -> int:
    state = torch.load(checkpoint_path, map_location="cpu")
    model_state = model.state_dict()
    compatible = {
        key: value
        for key, value in state.items()
        if key in model_state and tuple(model_state[key].shape) == tuple(value.shape)
    }
    model.load_state_dict(compatible, strict=False)
    return len(compatible)

model = build_convnext_tiny(len(class_names))
checkpoint_path = resolve_a3b_checkpoint()
if checkpoint_path is not None:
    loaded_count = load_compatible_weights(model, checkpoint_path)
    print(f"Loaded {loaded_count} compatible tensors from {checkpoint_path}")
else:
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    in_features = model.classifier[2].in_features
    model.classifier[2] = make_classifier_head(in_features, len(class_names))
    print("A3b checkpoint not found; started from ImageNet weights.")

for name, parameter in model.named_parameters():
    parameter.requires_grad = name.startswith("classifier.")

model = model.to(device)
optimizer = optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=CFG.HEAD_LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
best_checkpoint_path = RESULTS_DIR / "expanded_convnext_tiny_head_best.pth"

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

## 8. Training Loop

- Train the expanded head and keep the best validation checkpoint.
- Stop early if validation top-1 plateaus.
- Save training history after each epoch so partial runs remain useful.

In [ ]:
def run_epoch(loader: DataLoader, optimizer: Optional[optim.Optimizer] = None) -> dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.set_grad_enabled(is_train):
        for images, labels in tqdm(loader, leave=False, desc="train" if is_train else "eval"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += logits.argmax(dim=1).eq(labels).sum().item()
            total_count += batch_size
    return {"loss": total_loss / total_count, "top_1_accuracy": 100.0 * total_correct / total_count}

history = []
best_val_top_1 = -np.inf
patience_count = 0
for epoch in range(1, CFG.MAX_EPOCHS + 1):
    started = time.time()
    train_metrics = run_epoch(train_loader, optimizer)
    val_metrics = run_epoch(val_loader)
    scheduler.step(val_metrics["top_1_accuracy"])
    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_top_1_accuracy": train_metrics["top_1_accuracy"],
        "val_loss": val_metrics["loss"],
        "val_top_1_accuracy": val_metrics["top_1_accuracy"],
        "head_lr": optimizer.param_groups[0]["lr"],
        "elapsed_seconds": time.time() - started,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(RESULTS_DIR / "training_history.csv", index=False)
    print(json.dumps(row, indent=2))
    if val_metrics["top_1_accuracy"] > best_val_top_1:
        best_val_top_1 = val_metrics["top_1_accuracy"]
        patience_count = 0
        torch.save(model.state_dict(), best_checkpoint_path)
        print(f"Saved {best_checkpoint_path}")
    else:
        patience_count += 1
        if patience_count >= CFG.PATIENCE:
            print("Early stopping triggered.")
            break

## 9. Evaluation Utilities

- Reuse logits for top-k accuracy, calibration, source metrics, and prediction exports.
- Fit temperature scaling only on validation logits.
- Export metrics by source dataset to separate Food-101 regression from new-class behavior.

In [ ]:
def load_best_model() -> nn.Module:
    best_model = build_convnext_tiny(len(class_names))
    best_model.load_state_dict(torch.load(best_checkpoint_path, map_location=device))
    return best_model.to(device).eval()


def collect_predictions(eval_model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    logits_parts = []
    label_parts = []
    with torch.no_grad():
        for images, labels in tqdm(loader, leave=False, desc="predict"):
            images = images.to(device, non_blocking=True)
            logits_parts.append(eval_model(images).cpu())
            label_parts.append(labels.cpu())
    return torch.cat(logits_parts).numpy(), torch.cat(label_parts).numpy()


def topk_accuracy(logits: np.ndarray, labels: np.ndarray, k: int) -> float:
    topk = np.argsort(logits, axis=1)[:, -k:]
    return 100.0 * np.mean([label in row for label, row in zip(labels, topk)])


def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, bins: int = CFG.ECE_BINS) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = predictions == labels
    ece = 0.0
    edges = np.linspace(0.0, 1.0, bins + 1)
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidences > lower) & (confidences <= upper)
        if mask.any():
            ece += mask.mean() * abs(accuracies[mask].mean() - confidences[mask].mean())
    return float(ece)


def fit_temperature(logits: np.ndarray, labels: np.ndarray) -> float:
    logits_tensor = torch.tensor(logits, dtype=torch.float32, device=device)
    labels_tensor = torch.tensor(labels, dtype=torch.long, device=device)
    temperature = torch.ones(1, device=device, requires_grad=True)
    optimizer_temp = optim.LBFGS([temperature], lr=0.01, max_iter=50)

    def closure() -> torch.Tensor:
        optimizer_temp.zero_grad()
        loss = F.cross_entropy(logits_tensor / temperature.clamp(min=0.05), labels_tensor)
        loss.backward()
        return loss

    optimizer_temp.step(closure)
    return float(temperature.detach().clamp(min=0.05).cpu().item())


def metrics_from_logits(logits: np.ndarray, labels: np.ndarray, temperature: float = 1.0) -> dict[str, float]:
    scaled_logits = logits / temperature
    probs = F.softmax(torch.tensor(scaled_logits), dim=1).numpy()
    return {
        "top_1_accuracy": topk_accuracy(logits, labels, 1),
        "top_5_accuracy": topk_accuracy(logits, labels, min(CFG.TOP_K, logits.shape[1])),
        "ece": expected_calibration_error(probs, labels),
    }

best_model = load_best_model()
val_logits, val_labels = collect_predictions(best_model, val_loader)
test_logits, test_labels = collect_predictions(best_model, test_loader)
temperature = fit_temperature(val_logits, val_labels)

val_raw = metrics_from_logits(val_logits, val_labels, 1.0)
val_cal = metrics_from_logits(val_logits, val_labels, temperature)
test_raw = metrics_from_logits(test_logits, test_labels, 1.0)
test_cal = metrics_from_logits(test_logits, test_labels, temperature)

summary_df = pd.DataFrame([
    {"split": "val", "temperature": temperature, "top_1_accuracy": val_raw["top_1_accuracy"], "top_5_accuracy": val_raw["top_5_accuracy"], "ece_raw": val_raw["ece"], "ece_calibrated": val_cal["ece"]},
    {"split": "test", "temperature": temperature, "top_1_accuracy": test_raw["top_1_accuracy"], "top_5_accuracy": test_raw["top_5_accuracy"], "ece_raw": test_raw["ece"], "ece_calibrated": test_cal["ece"]},
])
summary_df.to_csv(RESULTS_DIR / "expanded_metrics.csv", index=False)

test_probs = F.softmax(torch.tensor(test_logits / temperature), dim=1).numpy()
test_pred_idx = test_probs.argmax(axis=1)
test_pred_labels = [class_names[idx] for idx in test_pred_idx]
test_true_labels = [class_names[idx] for idx in test_labels]

test_predictions_df = test_df.copy()
test_predictions_df["true_label"] = test_true_labels
test_predictions_df["pred_label"] = test_pred_labels
test_predictions_df["confidence"] = test_probs.max(axis=1)
test_predictions_df["is_correct"] = test_predictions_df["true_label"] == test_predictions_df["pred_label"]
test_predictions_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

source_metrics_df = (
    test_predictions_df.groupby("dataset")
    .agg(image_count=("path", "count"), top_1_accuracy=("is_correct", lambda values: 100.0 * values.mean()))
    .reset_index()
)
source_metrics_df.to_csv(RESULTS_DIR / "source_metrics.csv", index=False)

class_report_df = pd.DataFrame(classification_report(test_true_labels, test_pred_labels, output_dict=True)).T
class_report_df.to_csv(RESULTS_DIR / "test_class_report.csv")

print("Expanded metrics")
display(summary_df)
print("Source metrics")
display(source_metrics_df)
print(f"Outputs written to: {RESULTS_DIR}")